# ReAct

- 不断重复 Thought -> Action -> Observation 的循环
    - thought：思考。分析当前情况、分解任务、制定下一步计划，或者反思上一步的结果；
    - action：行动。决定采取的具体行动，通常是调用一个外部工具；
    - observation：观察。执行action后从外部工具返回的结果。

---

- 抽象工作流程：在每个时间步 t ,智能体的决策（即大预言模型 $\pi$ ）会根据初始问题 q 和之前所有步骤的“行动-观察”历史轨迹 $((a_1,o_1),...,(a_{t-1},o_{t-1}))$ ，来生成当前的思考 $th_t$ 和行动 $a_t$ ：
$$(th_t,a_t)=\pi (q, (a_1,o_1),...,(a_{t-1},o_{t-1}))$$
随后，环境中的工具 T 会执行行动 $a_t$ ，并返回一个新的观察结果 $o_t$：
$$o_t = T(a_t)$$
这个循环不断进行，将新的 $(a_t,o_t)$ 对追加到历史中，知道模型在思考 $th_t$ 中判断任务已完成。

---

- **优点**：
    1. 高可解释性：通过thought链，能清晰的知道agent每一步的心路历程；
    2. 动态规划和纠错能力：不是直接生成完整链式，而是“走一步看一步”，每一步都根据从外界获得的observation调整后续的thought和action；
    3. 与外部工具的执行能力结合
- **缺点**：
    1. 步进式的决策模式意味着缺乏一个全局的、长远的规划，可能导致陷入局部最优、原地打转。
    2. 循序渐进的特性，完成一个任务需要多次调用LLM，每次调用都伴随网络延迟和计算成本。

In [3]:
# Prompt template - ReAct

REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

请严格按照以下格式进行回应:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一:
- `{{tool_name}}[{{tool_input}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""

ReAct 核心是一个循环，不断地“格式化提示词 -> 调用LLM -> 执行动作 -> 整合结果” 直到任务完成或达到最大步数限制

In [23]:
import sys
sys.path.append(r"C:\Users\61075\PycharmProjects\learning\hello_agent")
from _LLM import AgentsLLM
from _Tool_search import search
from _ToolExecutor import ToolExecutor
import re
from dotenv import load_dotenv

load_dotenv(r"C:\Users\61075\PycharmProjects\learning\hello_agent\.env")

True

In [27]:
class ReActAgent:
    def __init__(self, llm_client: AgentsLLM, tool_executor: ToolExecutor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []

    @staticmethod
    def _parse_output(text: str):
        """解析LLM的输出，提取Thought和Action"""
        # Thought: 匹配到Action或文本末尾
        thought_match = re.search(
            r"Thought:\s*(.*?)(?=\nAction:|$)",
            text,
            re.DOTALL   # 让点符号(.)可以匹配换行符(\n)等所有符号
        )
        # Action: 匹配到文本末尾
        action_match = re.search(
            r"Action:\s*(.*?)$",
            text,
            re.DOTALL
        )
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action

    @staticmethod
    def _parse_action(action_text: str):
        """进一步解析Action字符串，提取工具名称和输入"""
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        if match:
            return match.group(1), match.group(2)
        return None, None

    def run(self, question: str):
        """运行agent来回答一个问题"""
        self.history = []   # 每次运行时重置历史记录
        current_step = 0

        while current_step < self.max_steps:
            current_step += 1
            print(f"--- current_step: {current_step} ---")

            # 1. 格式化提示词
            tools_desc = self.tool_executor.get_available_tools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=tools_desc,
                question=question,
                history=history_str
            )

            # 2. 调用LLM思考
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages)

            if not response_text:
                print("ERROR: LLM cannot connect")
                break

            # 3. 解析LLM的输出
            thought, action = self._parse_output(response_text)
            if thought:
                print(f"%%% THOUGHT: {thought}")
            if not action:
                print(f"WARNING: no action")
                break

            # 4. 执行Action
            if action.startswith("Finish"):
                # 如果是Finish指令，则无需调用工具，直接提取最终答案并结束
                final_answer = re.match(r"Finish\[(.*)\]", action).group(1)
                print(f"%%% Final answer: {final_answer}")
                return final_answer

            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                print(f"ERROR: invalid action: {action}")
                continue

            print(f"%%% ACTION: {tool_name}[{tool_input}]")

            tool_function = self.tool_executor.get_tool(tool_name)
            if not tool_function:
                observation = f"ERROR: no such tool: {tool_name}"
            else:
                observation = tool_function(tool_input) # 调用真实工具

            print(f"%%% OBSERVATION: {observation}")

            # 将本轮的Action和Observation添加到历史记录中
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        # 循环结束
        print("End, reached max steps")
        return None

In [28]:
llm = AgentsLLM()
tool_executor = ToolExecutor()

search_desc = "一个网页搜索引擎工具。当需要回答关于时事、事实以及在知识库中找不到的信息时，应使用此工具。"
tool_executor.register_tool("Search", search_desc, search)

agent = ReActAgent(
    llm_client=llm,
    tool_executor=tool_executor,
    max_steps=5
)

question = "苹果最新的手机是哪一款？它的主要卖点是什么？"
agent.run(question)

INFO: Tool 'Search' has been registered
--- current_step: 1 ---
*** 正在调用 llama-3.3-70b-versatile 模型...
*** 大语言模型响应成功：
Thought: 要找到苹果最新的手机型号及其主要卖点，我需要访问最新的信息，因为苹果公司经常发布新产品。由于我的知识可能已经过时，我应该使用网页搜索工具来获取最新的信息。

Action: Search[苹果最新手机型号及主要卖点]
%%% THOUGHT: 要找到苹果最新的手机型号及其主要卖点，我需要访问最新的信息，因为苹果公司经常发布新产品。由于我的知识可能已经过时，我应该使用网页搜索工具来获取最新的信息。
%%% ACTION: Search[苹果最新手机型号及主要卖点]
** 正在执行 SerpApi 网页搜索：苹果最新手机型号及主要卖点
%%% OBSERVATION: [1] 一图看懂苹果2025秋季新品发布会，八大新品卖点价格全汇总
本次发布会上，苹果带来了iPhone17、iPhone 17 Pro/Pro Max以及全新系列iPhone Air，还有AirPods Pro 3、Apple Watch Series 11、Apple Watch SE 3以及Apple ...

[2] 2026年苹果iPhone手机推荐哪一款性价比最高？（8000字选 ...
最新的iPhone共有iPhone 17、iPhone Air、iPhone 17 Pro以及iPhone 17 Pro Max四款机型。其中，属于基本款的iPhone 17配备6.3英寸120Hz高刷显示屏和A19芯片，相对平价且已 ...

[3] iPhone - 机型比较
比较各款iPhone 机型的功能和技术规格，包括iPhone 17 Pro、iPhone 17 Pro Max、iPhone Air、iPhone 17、iPhone 17e 以及更多机型。
--- current_step: 2 ---
*** 正在调用 llama-3.3-70b-versatile 模型...
*** 大语言模型响应成功：
Thought: 我已经收集到了一些关于苹果最新手机型号的信息，包括iPhone 17、iPhone 17 Pro/Pro Max、iPho

'iPhone 17、iPhone 17 Pro/Pro Max和iPhone Air是苹果最新的手机型号，主要卖点包括高刷显示屏、A19芯片等。'